# Кривые выживания скора (без фитов)

Все доступные проскоренные test-МК и exp события чекпоинта: кривые «доля с logit > порога» по классам и отношение exp/muatm vs доля muatm.


In [ ]:
CHECKPOINT = "261024_1202_da_nu_classifier_exp_full_E1_weights_z128_k10_dlr1e-2_lam0p003_delay2_h8s3_s42@best_da_model"


In [ ]:
import os, sys
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
ROOT = "/home/albert/Baikal2025"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + "/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")
os.chdir("/home/albert/Baikal2025/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")  # относительные пути как в остальных ноутбуках

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from excess.survival import load_events

EV, report = load_events(CHECKPOINT)
display(report)
exp, muatm, nuatm, nue2 = (EV["exp_full"], EV["muatm"],
                           EV["nuatm"], EV["nue2"])


In [ ]:
# Кривые выживания скора и отношение exp/muatm.
# Хиты и фиты не читались: population — все проскоренные test-МК и exp события.
LOGIT_MIN, LOGIT_MAX, N_POINTS = -6, 6, 400
MIN_N_SN_HITS, MIN_N_SN_STRINGS = 8, 3
Z = 1.96
MIN_MUATM_EVENTS = 20
N_ERRBARS = 24


def wilson(k, n, z=Z):
    """95% интервал Уилсона для биномиальной доли k/n."""
    k = np.asarray(k, dtype=float)
    p = k / n
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2.0 * n)) / denom
    half = z * np.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / denom
    return np.clip(centre - half, 0.0, 1.0), np.clip(centre + half, 0.0, 1.0)


frames = {"exp_full": exp, "muatm": muatm, "nuatm": nuatm, "nue2": nue2}
colors = {"exp_full": "#d62728", "muatm": "#1f77b4",
          "nuatm": "#2ca02c", "nue2": "#ff7f0e"}
grid = np.linspace(LOGIT_MIN, LOGIT_MAX, N_POINTS)

surv, lo, hi, total = {}, {}, {}, {}

fig, ax = plt.subplots(figsize=(8, 5))
for name, frame in frames.items():
    x = frame.loc[(frame.n_sn_hits >= MIN_N_SN_HITS)
                  & (frame.n_sn_strings >= MIN_N_SN_STRINGS), "logit"].to_numpy()
    x = x[np.isfinite(x)]
    if len(x) == 0:
        continue
    n = len(x)
    x.sort()
    k_above = n - np.searchsorted(x, grid, side="right")
    surv[name] = k_above / n
    lo[name], hi[name] = wilson(k_above, n)
    total[name] = n

    ax.plot(grid, surv[name], color=colors[name], lw=1.6,
            label=f"{name}  (N={n:,})")
    ax.fill_between(grid, np.maximum(lo[name], 1e-8), hi[name],
                    color=colors[name], alpha=0.15, lw=0)

ax.set_xlabel("порог по logit")
ax.set_ylabel("доля событий с logit > порога")
ax.set_yscale("log")
ax.set_ylim(1e-5, 1.0)
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8)
plt.show()

# --- отношение exp_full/muatm ---
if "exp_full" in surv and "muatm" in surv:
    s_e, n_e = surv["exp_full"], total["exp_full"]
    s_m, n_m = surv["muatm"], total["muatm"]
    ok = (s_m > 0) & (s_e > 0) & (s_m * n_m >= MIN_MUATM_EVENTS)
    x_m = s_m[ok]
    ratio = s_e[ok] / s_m[ok]
    sig_e = np.sqrt(s_e * (1.0 - s_e) / n_e)[ok]
    sig_m = np.sqrt(s_m * (1.0 - s_m) / n_m)[ok]
    sig_ratio = ratio * np.sqrt((sig_e / s_e[ok]) ** 2
                                + (sig_m / s_m[ok]) ** 2)
    x_lo, x_hi = lo["muatm"][ok], hi["muatm"][ok]

    order = np.argsort(x_m)
    xs, rs, sr = x_m[order], ratio[order], sig_ratio[order]
    xl, xh = x_lo[order], x_hi[order]

    fig2, ax2 = plt.subplots(figsize=(7, 5))
    ax2.plot(xs, rs, color="#7b52ab", lw=1.8)
    ax2.fill_between(xs, np.maximum(rs - sr, 1e-8), rs + sr,
                     color="#7b52ab", alpha=0.15, lw=0)
    ax2.axhline(1.0, color="grey", ls=":", lw=1)
    sel = np.unique(np.linspace(0, len(xs) - 1, N_ERRBARS).astype(int))
    ax2.errorbar(xs[sel], rs[sel],
                 xerr=np.vstack([xs[sel] - xl[sel], xh[sel] - xs[sel]]),
                 yerr=sr[sel], fmt="none", ecolor="#555555",
                 elinewidth=0.8, capsize=2, alpha=0.65)
    ax2.set_xscale("log")
    ax2.set_yscale("log")
    ax2.set_xlim(1e-4, None)
    ax2.set_xlabel("доля muatm с logit > порога  (95% интервал Уилсона)")
    ax2.set_ylabel("доля exp_full / доля muatm  (±1σ, дельта-метод)")
    ax2.grid(alpha=0.2, which="both")
    plt.show()
